Deploy com Gradio
Até aqui, foi construido todo o pipeline: carregou CTs, montou o dataset, treinou o modelo e avaliou o desempenho. Agora falta a cereja do bolo: colocar o modelo numa interface web que qualquer pessoa consiga usar.

A ideia e simples: o usuario faz upload de um CT scan (arquivos .mhd e .raw), o modelo classifica todos os candidatos a nodulo daquele CT, e a interface mostra os resultados numa tabela e numa visualizacao.

Pra isso, a gente vai usar o Gradio -- uma biblioteca Python que transforma qualquer funcao em interface web com poucas linhas de codigo.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import gradio as gr
import SimpleITK as sitk

sys.path.insert(0, str(Path("../src")))
from luna_data import load_candidates, xyz_to_irc, XYZ
from model import LunaModel
from inference import load_model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
model, info = load_model("../checkpoints/luna_model_best.pt", device)
print(f"Modelo carregado: epoca {info['epoch']}, F1={info['best_f1']:.4f}")

In [ ]:
all_candidates = load_candidates(require_on_disk=False)
n_pos = sum(1 for c in all_candidates if c.is_nodule)
print(f"Candidatos carregados: {len(all_candidates)} ({n_pos} nodulos)")

Repare que a gente carregou os candidatos com require_on_disk=False. Nos notebooks anteriores, a gente filtrava so os CTs que estavam no disco. Aqui, a gente precisa de todos os candidatos do CSV, porque o usuario pode fazer upload de qualquer CT do LUNA16.

Gradio em 5 minutos
Antes de montar a interface completa, vamos ver como o Gradio funciona com um exemplo minimo. A ideia e: voce escreve uma funcao Python, e o Gradio gera a interface automaticamente.

In [ ]:
def saudacao(nome):
    return f"Ola, {nome}! Bem-vindo ao detector de nodulos."

demo_simples = gr.Interface(
    fn=saudacao,
    inputs=gr.Textbox(label="Seu nome"),
    outputs=gr.Textbox(label="Resposta"),
)
demo_simples.launch(inline=True, height=400)

So isso. Tres linhas e voce tem uma interface web funcionando. O gr.Interface recebe a funcao, define os inputs e outputs, e o launch() sobe o servidor. Vamos fechar esse demo e partir pro negocio de verdade.

In [ ]:
demo_simples.close()

Funcao de classificacao para upload
No notebook 08, a gente criou a funcao classify_ct() que recebe um series_uid e procura o arquivo .mhd no diretorio de dados. Pra interface com upload, a gente precisa de uma versao diferente que:

Recebe os paths dos arquivos que o usuario fez upload
Le o CT direto desses paths com SimpleITK
Descobre o series_uid pelo nome do arquivo
Busca os candidatos no CSV e classifica
Uma limitacao importante: isso so funciona com CTs do LUNA16, porque a gente precisa das coordenadas dos candidatos que estao no candidates.csv. Pra um CT aleatorio que nao esta no dataset, a gente nao teria onde procurar.

In [ ]:
def classify_uploaded_ct(mhd_path, raw_path, candidates_list, model, device,
                         batch_size=64):
    """Classifica candidatos de um CT a partir de arquivos .mhd/.raw."""
    import shutil
    import tempfile

    # SimpleITK precisa dos dois arquivos no mesmo diretorio
    tmpdir = tempfile.mkdtemp()
    mhd_dst = Path(tmpdir) / Path(mhd_path).name
    raw_dst = Path(tmpdir) / Path(raw_path).name
    shutil.copy2(mhd_path, mhd_dst)
    shutil.copy2(raw_path, raw_dst)

    # O series_uid e o nome do arquivo .mhd sem extensao
    series_uid = mhd_dst.stem

    # Filtra candidatos desse CT
    candidates = [c for c in candidates_list if c.series_uid == series_uid]
    if not candidates:
        shutil.rmtree(tmpdir)
        return [], None, None

    # Le o CT com SimpleITK
    ct_mhd = sitk.ReadImage(str(mhd_dst))
    hu_a = np.array(sitk.GetArrayFromImage(ct_mhd), dtype=np.float32)
    hu_a.clip(-1000, 1000, hu_a)

    origin_xyz = XYZ(*ct_mhd.GetOrigin())
    vx_size_xyz = XYZ(*ct_mhd.GetSpacing())
    direction_a = np.array(ct_mhd.GetDirection()).reshape(3, 3)

    shutil.rmtree(tmpdir)

    # Extrai crops e classifica
    crops = []
    crop_size = (32, 48, 48)
    for c in candidates:
        center_irc = xyz_to_irc(c.center_xyz, origin_xyz, vx_size_xyz, direction_a)
        slices = []
        for axis, center_val in enumerate(center_irc):
            start = int(round(center_val - crop_size[axis] / 2))
            end = int(start + crop_size[axis])
            if start < 0:
                start, end = 0, int(crop_size[axis])
            if end > hu_a.shape[axis]:
                end = hu_a.shape[axis]
                start = int(end - crop_size[axis])
            slices.append(slice(start, end))
        crops.append(hu_a[tuple(slices)])

    crops_t = torch.from_numpy(np.stack(crops)).float().unsqueeze(1)

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(crops_t), batch_size):
            batch = crops_t[start:start + batch_size].to(device)
            _, probs = model(batch)
            all_probs.extend(probs[:, 1].cpu().numpy())

    results = []
    for c, prob, crop in zip(candidates, all_probs, crops):
        results.append({
            "series_uid": c.series_uid,
            "center_xyz": c.center_xyz,
            "probability": float(prob),
            "is_nodule": c.is_nodule,
            "crop": crop,
        })

    results.sort(key=lambda r: r["probability"], reverse=True)

    ct_info = {
        "hu_a": hu_a,
        "origin_xyz": origin_xyz,
        "vx_size_xyz": vx_size_xyz,
        "direction_a": direction_a,
    }
    return results, series_uid, ct_info

Vamos testar com um CT local pra ver se funciona.

In [ ]:
# Pega um CT que tem nodulos pra teste
test_uid = None
data_dir = Path("../data/luna")
for c in all_candidates:
    if c.is_nodule:
        mhd_matches = list(data_dir.rglob(f"{c.series_uid}.mhd"))
        if mhd_matches:
            test_uid = c.series_uid
            test_mhd = str(mhd_matches[0])
            test_raw = test_mhd.replace(".mhd", ".raw")
            break

print(f"CT de teste: {test_uid}")
print(f"Arquivo: {test_mhd}")

In [ ]:
results, uid, ct_info = classify_uploaded_ct(
    test_mhd, test_raw, all_candidates, model, device,
)
print(f"CT: {uid}")
print(f"Candidatos classificados: {len(results)}")
print(f"\nTop 5 por probabilidade:")
for r in results[:5]:
    label = "nodulo" if r["is_nodule"] else "nao-nod"
    print(f"  prob={r['probability']:.4f}  [{label}]  xyz=({r['center_xyz'][0]:.1f}, {r['center_xyz'][1]:.1f}, {r['center_xyz'][2]:.1f})")

Top 5 por probabilidade:
  prob=1.0000  [nodulo]  xyz=(66.8, 81.9, -121.8)
  prob=0.9999  [nodulo]  xyz=(67.6, 85.0, -109.8)
  prob=0.9999  [nodulo]  xyz=(75.2, 79.5, -121.4)
  prob=0.9999  [nodulo]  xyz=(66.1, 91.4, -122.0)
  prob=0.9996  [nodulo]  xyz=(64.1, 85.0, -95.3)

Tabela e visualizacao
Numa aplicacao real, nao existe ground truth -- o modelo recebe um CT e diz o que encontrou. Entao a gente vai:

Filtrar so os candidatos suspeitos (probabilidade acima de um threshold)
Mostrar uma tabela limpa (sem coluna de ground truth)
Montar um grid de imagens com todos os suspeitos, em linhas de 3

In [ ]:
def build_dataframe(suspects):
    """Monta DataFrame apenas com os candidatos suspeitos."""
    rows = []
    for i, r in enumerate(suspects):
        rows.append({
            "#": i + 1,
            "Probabilidade": f"{r['probability']:.4f}",
            "X": f"{r['center_xyz'][0]:.1f}",
            "Y": f"{r['center_xyz'][1]:.1f}",
            "Z": f"{r['center_xyz'][2]:.1f}",
        })
    return pd.DataFrame(rows)

In [ ]:
def build_figure(suspects):
    """Monta grid com fatia axial central de cada suspeito (3 por linha)."""
    n = len(suspects)
    if n == 0:
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.text(0.5, 0.5, "Nenhum candidato suspeito",
                ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes).flatten()

    for i in range(n):
        r = suspects[i]
        crop = r["crop"]
        axes[i].imshow(crop[crop.shape[0] // 2], cmap="gray")
        axes[i].set_title(f"#{i+1}  prob={r['probability']:.3f}", fontsize=11)
        axes[i].axis("off")

    for i in range(n, len(axes)):
        axes[i].axis("off")

    plt.suptitle("Candidatos suspeitos", fontsize=13)
    plt.tight_layout()
    return fig

In [ ]:
threshold = 0.5
suspects = [r for r in results if r["probability"] >= threshold]
print(f"Suspeitos (prob >= {threshold}): {len(suspects)}")

df = build_dataframe(suspects)
df

In [ ]:
fig = build_figure(suspects)
plt.show()

Interface completa
Agora a gente junta tudo numa interface Gradio. O fluxo e:

O usuario faz upload de 2 arquivos (.mhd e .raw do mesmo CT)
Clica em "Classificar"
A interface mostra: resumo em texto, tabela de resultados, e grid visual
A gente usa gr.Blocks em vez de gr.Interface porque permite mais controle sobre o layout.

In [ ]:
def analyze_ct(files, threshold):
    """Funcao principal da interface Gradio."""
    if files is None or len(files) < 2:
        return "Faca upload de 2 arquivos: um .mhd e um .raw do mesmo CT.", None, None

    mhd_path = None
    raw_path = None
    for f in files:
        name = f if isinstance(f, str) else f.name
        if name.endswith(".mhd"):
            mhd_path = name
        elif name.endswith(".raw"):
            raw_path = name

    if mhd_path is None or raw_path is None:
        return "Nao encontrei os arquivos .mhd e .raw. Verifique o upload.", None, None

    results, uid, ct_info = classify_uploaded_ct(
        mhd_path, raw_path, all_candidates, model, device,
    )

    if not results:
        return (
            "Nenhum candidato encontrado para este CT.\n"
            "Isso pode significar que o CT nao faz parte do dataset LUNA16, "
            "ou que o series_uid nao esta no candidates.csv.",
            None, None,
        )

    suspects = [r for r in results if r["probability"] >= threshold]

    resumo = (
        f"CT: {uid}\n"
        f"Candidatos analisados: {len(results)}\n"
        f"Suspeitos (prob >= {threshold}): {len(suspects)}\n"
    )
    if suspects:
        resumo += f"Maior probabilidade: {suspects[0]['probability']:.4f}"
    else:
        resumo += "Nenhuma regiao suspeita encontrada."

    df = build_dataframe(suspects)
    fig = build_figure(suspects)

    return resumo, df, fig

In [ ]:
with gr.Blocks(title="Detector de Nodulos Pulmonares") as demo:
    gr.Markdown(
        "## Detector de Nodulos Pulmonares (LUNA16)\n"
        "Faca upload dos arquivos `.mhd` e `.raw` de um CT scan do dataset LUNA16. "
        "O modelo analisa todas as regioes candidatas e mostra as suspeitas.\n\n"
        "**Importante**: so funciona com CTs do LUNA16 (precisa das coordenadas do `candidates.csv`)."
    )

    with gr.Row():
        file_input = gr.File(
            file_count="multiple",
            file_types=[".mhd", ".raw"],
            label="Arquivos do CT (.mhd e .raw)",
        )
        threshold_slider = gr.Slider(
            minimum=0.1, maximum=0.95, value=0.5, step=0.05,
            label="Threshold",
        )

    btn = gr.Button("Classificar", variant="primary")

    resumo_output = gr.Textbox(label="Resumo", lines=4)
    df_output = gr.Dataframe(label="Regioes suspeitas")
    plot_output = gr.Plot(label="Visualizacao dos suspeitos")

    btn.click(
        fn=analyze_ct,
        inputs=[file_input, threshold_slider],
        outputs=[resumo_output, df_output, plot_output],
    )

demo.launch(inline=True, height=700)

In [ ]:
demo.close()

Exportando o app.py
Agora a gente empacota tudo num script app.py que roda independente do notebook. O script:

Carrega o modelo e os candidatos na inicializacao
Sobe a interface Gradio
Pode ser executado com python app.py

In [ ]:
%%writefile ../app.py
"""Detector de Nodulos Pulmonares (LUNA16) - Interface Gradio."""

import shutil
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import gradio as gr

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from luna_data import load_candidates, xyz_to_irc, XYZ
from model import LunaModel
from inference import load_model

# --- Inicializacao ---

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL, MODEL_INFO = load_model(ROOT / "checkpoints" / "luna_model_best.pt", DEVICE)
ALL_CANDIDATES = load_candidates(require_on_disk=False)

print(f"Modelo carregado: epoca {MODEL_INFO['epoch']}, F1={MODEL_INFO['best_f1']:.4f}")
print(f"Candidatos: {len(ALL_CANDIDATES)}")
print(f"Device: {DEVICE}")


# --- Funcoes de classificacao ---

def classify_uploaded_ct(mhd_path, raw_path, batch_size=64):
    """Classifica candidatos de um CT a partir de arquivos .mhd/.raw."""
    tmpdir = tempfile.mkdtemp()
    mhd_dst = Path(tmpdir) / Path(mhd_path).name
    raw_dst = Path(tmpdir) / Path(raw_path).name
    shutil.copy2(mhd_path, mhd_dst)
    shutil.copy2(raw_path, raw_dst)

    series_uid = mhd_dst.stem

    candidates = [c for c in ALL_CANDIDATES if c.series_uid == series_uid]
    if not candidates:
        shutil.rmtree(tmpdir)
        return [], None

    ct_mhd = sitk.ReadImage(str(mhd_dst))
    hu_a = np.array(sitk.GetArrayFromImage(ct_mhd), dtype=np.float32)
    hu_a.clip(-1000, 1000, hu_a)

    origin_xyz = XYZ(*ct_mhd.GetOrigin())
    vx_size_xyz = XYZ(*ct_mhd.GetSpacing())
    direction_a = np.array(ct_mhd.GetDirection()).reshape(3, 3)

    shutil.rmtree(tmpdir)

    crop_size = (32, 48, 48)
    crops = []
    for c in candidates:
        center_irc = xyz_to_irc(c.center_xyz, origin_xyz, vx_size_xyz, direction_a)
        slices = []
        for axis, center_val in enumerate(center_irc):
            start = int(round(center_val - crop_size[axis] / 2))
            end = int(start + crop_size[axis])
            if start < 0:
                start, end = 0, int(crop_size[axis])
            if end > hu_a.shape[axis]:
                end = hu_a.shape[axis]
                start = int(end - crop_size[axis])
            slices.append(slice(start, end))
        crops.append(hu_a[tuple(slices)])

    crops_t = torch.from_numpy(np.stack(crops)).float().unsqueeze(1)

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(crops_t), batch_size):
            batch = crops_t[start:start + batch_size].to(DEVICE)
            _, probs = MODEL(batch)
            all_probs.extend(probs[:, 1].cpu().numpy())

    results = []
    for c, prob, crop in zip(candidates, all_probs, crops):
        results.append({
            "series_uid": c.series_uid,
            "center_xyz": c.center_xyz,
            "probability": float(prob),
            "is_nodule": c.is_nodule,
            "crop": crop,
        })

    results.sort(key=lambda r: r["probability"], reverse=True)
    return results, series_uid


# --- Funcoes de apresentacao ---

def build_dataframe(suspects):
    """Monta DataFrame apenas com os candidatos suspeitos."""
    rows = []
    for i, r in enumerate(suspects):
        rows.append({
            "#": i + 1,
            "Probabilidade": f"{r['probability']:.4f}",
            "X": f"{r['center_xyz'][0]:.1f}",
            "Y": f"{r['center_xyz'][1]:.1f}",
            "Z": f"{r['center_xyz'][2]:.1f}",
        })
    return pd.DataFrame(rows)


def build_figure(suspects):
    """Monta grid com fatia axial central de cada suspeito (3 por linha)."""
    n = len(suspects)
    if n == 0:
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.text(0.5, 0.5, "Nenhum candidato suspeito",
                ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes).flatten()

    for i in range(n):
        r = suspects[i]
        crop = r["crop"]
        axes[i].imshow(crop[crop.shape[0] // 2], cmap="gray")
        axes[i].set_title(f"#{i+1}  prob={r['probability']:.3f}", fontsize=11)
        axes[i].axis("off")

    for i in range(n, len(axes)):
        axes[i].axis("off")

    plt.suptitle("Candidatos suspeitos", fontsize=13)
    plt.tight_layout()
    return fig


# --- Funcao principal ---

def analyze_ct(files, threshold):
    """Funcao principal da interface Gradio."""
    if files is None or len(files) < 2:
        return "Faca upload de 2 arquivos: um .mhd e um .raw do mesmo CT.", None, None

    mhd_path = None
    raw_path = None
    for f in files:
        name = f if isinstance(f, str) else f.name
        if name.endswith(".mhd"):
            mhd_path = name
        elif name.endswith(".raw"):
            raw_path = name

    if mhd_path is None or raw_path is None:
        return "Nao encontrei os arquivos .mhd e .raw. Verifique o upload.", None, None

    results, uid = classify_uploaded_ct(mhd_path, raw_path)

    if not results:
        return (
            "Nenhum candidato encontrado para este CT.\n"
            "Isso pode significar que o CT nao faz parte do dataset LUNA16, "
            "ou que o series_uid nao esta no candidates.csv.",
            None, None,
        )

    suspects = [r for r in results if r["probability"] >= threshold]

    resumo = (
        f"CT: {uid}\n"
        f"Candidatos analisados: {len(results)}\n"
        f"Suspeitos (prob >= {threshold}): {len(suspects)}\n"
    )
    if suspects:
        resumo += f"Maior probabilidade: {suspects[0]['probability']:.4f}"
    else:
        resumo += "Nenhuma regiao suspeita encontrada."

    df = build_dataframe(suspects)
    fig = build_figure(suspects)

    return resumo, df, fig


# --- Interface Gradio ---

with gr.Blocks(title="Detector de Nodulos Pulmonares") as demo:
    gr.Markdown(
        "## Detector de Nodulos Pulmonares (LUNA16)\n"
        "Faca upload dos arquivos `.mhd` e `.raw` de um CT scan do dataset LUNA16. "
        "O modelo analisa todas as regioes candidatas e mostra as suspeitas.\n\n"
        "**Importante**: so funciona com CTs do LUNA16 (precisa das coordenadas do `candidates.csv`)."
    )

    with gr.Row():
        file_input = gr.File(
            file_count="multiple",
            file_types=[".mhd", ".raw"],
            label="Arquivos do CT (.mhd e .raw)",
        )
        threshold_slider = gr.Slider(
            minimum=0.1, maximum=0.95, value=0.5, step=0.05,
            label="Threshold",
        )

    btn = gr.Button("Classificar", variant="primary")

    resumo_output = gr.Textbox(label="Resumo", lines=4)
    df_output = gr.Dataframe(label="Regioes suspeitas")
    plot_output = gr.Plot(label="Visualizacao dos suspeitos")

    btn.click(
        fn=analyze_ct,
        inputs=[file_input, threshold_slider],
        outputs=[resumo_output, df_output, plot_output],
    )

if __name__ == "__main__":
    demo.launch(share=True)

Como usar
Pra rodar a aplicacao fora do notebook:

python app.py
Isso sobe um servidor local (geralmente em http://127.0.0.1:7860). Abra no navegador, arraste os arquivos .mhd e .raw de um CT, e clique em "Classificar".

Se quiser compartilhar com alguem (gerar um link publico temporario):

demo.launch(share=True)
O Gradio cria um tunel e te da um link tipo https://xxxxx.gradio.live que funciona por 72 horas.

Recapitulando o pipeline
Ao longo das ultimas aulas, a gente construiu um pipeline completo de deteccao de nodulos pulmonares:

Dados: baixamos o LUNA16, exploramos os CSVs, unificamos as fontes de dados
Dataset: carregamos CTs com SimpleITK, convertemos coordenadas, extraimos crops 3D
Modelo: CNN 3D com 4 blocos convolucionais, ~222K parametros
Treino: batches balanceados, augmentation 3D, 20 epocas no Colab
Avaliacao: curvas ROC/PR, analise de erros, escolha de threshold
Deploy: interface Gradio com upload de CT e visualizacao dos resultados
De dados brutos a uma aplicacao funcional -- esse e o ciclo completo de um projeto de Deep Learning aplicado.